# Bottle Cap Detection - Live Validation

This notebook loads the trained YOLO model and performs real-time bottle cap detection on the video file `caps.mov` using OpenCV.

## Features
- Load trained YOLOv11 model
- Process video frame by frame
- Draw bounding boxes and confidence scores in real-time
- Save annotated video output
- Display detection statistics

## Prerequisites
- Trained YOLO model (`yolo_models/bottle_cap_detection/weights/best.pt`)
- Video file (`caps.mov`)
- Required packages: ultralytics, opencv-python, numpy, matplotlib

## Step 1: Import Libraries and Setup

In [ ]:
# Install required packages if needed
import sys
import subprocess

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from ultralytics import YOLO
    import os
    from pathlib import Path
    import time
    from IPython.display import Video, display
    print("✅ All required packages are available")
except ImportError as e:
    print(f"Installing missing package: {e}")
    missing_packages = ["opencv-python", "ultralytics", "numpy", "matplotlib"]
    for package in missing_packages:
        try:
            install_package(package)
        except:
            print(f"Failed to install {package}")
    
    # Re-import after installation
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from ultralytics import YOLO
    import os
    from pathlib import Path
    import time
    from IPython.display import Video, display
    print("✅ Packages installed and imported successfully")

## Step 2: Configuration and File Paths

In [ ]:
# Configuration
MODEL_PATH = "yolo_models/bottle_cap_detection/weights/best.pt"
VIDEO_PATH = "caps.mov"
OUTPUT_VIDEO_PATH = "caps_annotated.mp4"
OUTPUT_FRAMES_DIR = "validation_frames"

# Detection settings
CONFIDENCE_THRESHOLD = 0.9  # Minimum confidence for detections
SAVE_FRAMES = False  # Save individual annotated frames
SAVE_VIDEO = False   # Save annotated video
DISPLAY_LIVE = True  # Set to True for live display (might be slow in Jupyter)

# Tracking settings
ENABLE_TRACKING = True  # Enable YOLO tracking for unique object IDs
TRACKER_TYPE = "bytetrack.yaml"  # Options: "bytetrack.yaml", "botsort.yaml"
TRACK_ZONE_X_MIN = 0.2  # Only track objects in middle 60% of frame (20% to 80% x-coordinate)
TRACK_ZONE_X_MAX = 0.8  # This helps avoid tracking issues when caps enter/exit frame edges

# Video processing settings
DOWNSCALE_FACTOR = 1.0  # Scale factor for video processing (1.0 = original, 0.5 = half size)
TARGET_HEIGHT = 480     # Alternative: set target height (will override DOWNSCALE_FACTOR if not None)
USE_TARGET_HEIGHT = False  # If True, use TARGET_HEIGHT instead of DOWNSCALE_FACTOR

# Class names
CLASS_NAMES = ["bottle_cap"]

# Base colors for different track IDs (BGR format for OpenCV)
BASE_COLORS = [
    (0, 255, 0),    # Green
    (255, 0, 0),    # Blue
    (0, 0, 255),    # Red
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255),  # Yellow
    (128, 0, 128),  # Purple
    (255, 165, 0),  # Orange
    (0, 128, 255),  # Light Blue
    (128, 255, 0),  # Lime
    (255, 192, 203), # Pink
    (165, 42, 42),  # Brown
    (128, 128, 128), # Gray
    (255, 20, 147), # Deep Pink
    (0, 191, 255),  # Deep Sky Blue
]

print(f"Model path: {MODEL_PATH}")
print(f"Video path: {VIDEO_PATH}")
print(f"Output video: {OUTPUT_VIDEO_PATH}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Tracking enabled: {ENABLE_TRACKING}")
if ENABLE_TRACKING:
    print(f"Tracker type: {TRACKER_TYPE}")

# Create output directory
if SAVE_FRAMES:
    Path(OUTPUT_FRAMES_DIR).mkdir(exist_ok=True)
    print(f"Output frames directory: {OUTPUT_FRAMES_DIR}")

In [ ]:
# Quick directory check and alternative video file suggestions
print("🔍 Checking current directory for video files...")

# List all video files in current directory
current_dir = "."
video_extensions = ['.mov', '.mp4', '.avi', '.mkv', '.webm', '.flv', '.wmv']

video_files = []
for file in os.listdir(current_dir):
    if any(file.lower().endswith(ext) for ext in video_extensions):
        file_path = os.path.join(current_dir, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        video_files.append((file, file_size))

if video_files:
    print(f"📹 Found {len(video_files)} video file(s):")
    for file, size in video_files:
        print(f"   • {file} ({size:.2f} MB)")
    
    # Suggest alternatives if caps.mov is not found or corrupted
    if not os.path.exists(VIDEO_PATH):
        print(f"\n💡 '{VIDEO_PATH}' not found. Alternative options:")
        for file, size in video_files:
            print(f"   Change VIDEO_PATH to: '{file}'")
else:
    print("❌ No video files found in current directory")
    print("\n💡 Solutions:")
    print("   1. Copy your video file to this directory")
    print("   2. Update VIDEO_PATH with the correct path")
    print("   3. Use a different video file for testing")

# Check if the specified video path exists
print(f"\n🎯 Checking specified video path: {VIDEO_PATH}")
if os.path.exists(VIDEO_PATH):
    file_size = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
    print(f"✅ File exists ({file_size:.2f} MB)")
else:
    print(f"❌ File not found at: {VIDEO_PATH}")
    
    # Look for the file in other common locations
    search_paths = [
        "../caps.mov",
        "../cdm-training/caps.mov", 
        "./frames/caps.mov",
        "~/Downloads/caps.mov"
    ]
    
    print(f"\n🔍 Searching in common locations...")
    for path in search_paths:
        full_path = os.path.expanduser(path)
        if os.path.exists(full_path):
            print(f"✅ Found at: {full_path}")
            print(f"   Update VIDEO_PATH to: '{full_path}'")
            break
    else:
        print(f"❌ Video file not found in common locations")

## Step 3: Load YOLO Model

In [ ]:
# Load the trained YOLO model
if os.path.exists(MODEL_PATH):
    print(f"Loading YOLO model from: {MODEL_PATH}")
    model = YOLO(MODEL_PATH)
    print("✅ Model loaded successfully!")
    
    # Display model info
    print(f"\nModel Information:")
    model.info()
    
else:
    print(f"❌ Model file not found: {MODEL_PATH}")
    print("Please ensure the model training was completed and the file exists.")
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

## Step 4: Load and Analyze Video

In [ ]:
# Enhanced video file diagnostics and corruption check
import hashlib

def check_video_integrity(video_path):
    """
    Comprehensive video file integrity and corruption check.
    """
    print(f"🔍 Diagnosing video file: {video_path}")
    
    # 1. Basic file existence and size check
    if not os.path.exists(video_path):
        print(f"❌ Video file not found: {video_path}")
        return False
    
    file_size = os.path.getsize(video_path) / (1024 * 1024)  # MB
    print(f"📁 File size: {file_size:.2f} MB")
    
    if file_size < 0.1:  # Less than 100KB
        print(f"⚠️  File size is very small ({file_size:.2f} MB), might be corrupted")
    
    # 2. Try to open with OpenCV
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"❌ OpenCV cannot open the video file")
        cap.release()
        return False
    
    # 3. Get basic video properties
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = frame_count / fps if fps > 0 else 0
        
        print(f"📹 Video properties:")
        print(f"   Resolution: {width}x{height}")
        print(f"   FPS: {fps}")
        print(f"   Total frames: {frame_count}")
        print(f"   Duration: {duration:.2f} seconds")
        
        # Check for invalid properties
        if fps <= 0 or frame_count <= 0 or width <= 0 or height <= 0:
            print(f"❌ Invalid video properties detected")
            cap.release()
            return False
            
    except Exception as e:
        print(f"❌ Error reading video properties: {e}")
        cap.release()
        return False
    
    # 4. Try to read first few frames
    print(f"🎞️  Testing frame reading...")
    frames_read = 0
    frames_failed = 0
    test_frames = min(10, frame_count)  # Test first 10 frames or all if less
    
    for i in range(test_frames):
        ret, frame = cap.read()
        if ret:
            frames_read += 1
            # Check if frame data is valid
            if frame is None or frame.size == 0:
                frames_failed += 1
        else:
            frames_failed += 1
    
    cap.release()
    
    success_rate = frames_read / test_frames * 100
    print(f"   Frames read successfully: {frames_read}/{test_frames} ({success_rate:.1f}%)")
    
    if success_rate < 50:
        print(f"❌ High frame read failure rate, video likely corrupted")
        return False
    elif success_rate < 100:
        print(f"⚠️  Some frames failed to read, video might have issues")
    else:
        print(f"✅ All test frames read successfully")
    
    # 5. File format detection
    try:
        import subprocess
        result = subprocess.run(['file', video_path], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            file_info = result.stdout.strip()
            print(f"📄 File type: {file_info}")
        else:
            print(f"⚠️  Could not determine file type")
    except:
        print(f"⚠️  File type detection not available")
    
    return True

# Run video diagnostics
video_is_valid = check_video_integrity(VIDEO_PATH)

if not video_is_valid:
    print(f"\n💡 Troubleshooting suggestions:")
    print(f"   1. Re-download or re-copy the video file")
    print(f"   2. Check if the file was completely transferred")
    print(f"   3. Try opening the file in a video player (VLC, etc.)")
    print(f"   4. Convert the video to a different format (mp4, avi)")
    print(f"   5. Check available disk space")
    raise ValueError(f"Video file appears to be corrupted: {VIDEO_PATH}")
else:
    print(f"\n✅ Video file appears to be valid and ready for processing!")

## Step 4b: Final Video Loading

In [ ]:
# Load video and get properties
if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    
    if cap.isOpened():
        # Get video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = frame_count / fps
        
        # Calculate processing dimensions
        if USE_TARGET_HEIGHT and TARGET_HEIGHT:
            # Scale based on target height
            scale_factor = TARGET_HEIGHT / height
            process_width = int(width * scale_factor)
            process_height = TARGET_HEIGHT
        else:
            # Use downscale factor
            process_width = int(width * DOWNSCALE_FACTOR)
            process_height = int(height * DOWNSCALE_FACTOR)
        
        print(f"✅ Video loaded successfully!")
        print(f"Original resolution: {width}x{height}")
        print(f"Processing resolution: {process_width}x{process_height}")
        print(f"Scale factor: {process_width/width:.2f}")
        print(f"FPS: {fps}")
        print(f"Total frames: {frame_count}")
        print(f"Duration: {duration:.2f} seconds")
        
        cap.release()
    else:
        print(f"❌ Could not open video file: {VIDEO_PATH}")
        raise ValueError(f"Could not open video: {VIDEO_PATH}")
else:
    print(f"❌ Video file not found: {VIDEO_PATH}")
    raise FileNotFoundError(f"Video file not found: {VIDEO_PATH}")

In [ ]:
# Utility classes for tracking and line crossing detection

class LineCrossingDetector:
    """
    Detects when tracked objects cross the middle vertical line of the frame.
    Logs crossing events with timestamps and direction.
    """
    
    def __init__(self, frame_width, frame_height, fps):
        self.frame_width = frame_width
        self.frame_height = frame_height
        self.fps = fps
        self.middle_line_x = frame_width // 2  # Middle vertical line
        
        # Track object positions and crossings
        self.object_positions = {}  # object_id -> {'x': last_x, 'side': 'left'/'right'}
        self.crossing_events = []   # List of crossing events
        
        print(f"🚦 Line crossing detector initialized - middle line at x={self.middle_line_x}")
    
    def update_position(self, object_id, x, y, timestamp):
        """
        Update an object's position and detect line crossings.
        
        Args:
            object_id: Unique identifier for the object
            x, y: Current position of the object center
            timestamp: Timestamp in seconds
        """
        current_side = 'left' if x < self.middle_line_x else 'right'
        
        # Check if this is a new object
        if object_id not in self.object_positions:
            self.object_positions[object_id] = {
                'x': x,
                'y': y,
                'side': current_side,
                'last_timestamp': timestamp
            }
            return
        
        # Get previous position
        prev_data = self.object_positions[object_id]
        prev_side = prev_data['side']
        prev_x = prev_data['x']
        
        # Check for line crossing
        if prev_side != current_side:
            # Determine crossing direction
            if prev_side == 'left' and current_side == 'right':
                direction = 'left_to_right'
                direction_symbol = '→'
            else:
                direction = 'right_to_left'
                direction_symbol = '←'
            
            # Log the crossing event
            crossing_event = {
                'object_id': object_id,
                'timestamp': timestamp,
                'direction': direction,
                'previous_x': prev_x,
                'current_x': x,
                'y_position': y
            }
            
            self.crossing_events.append(crossing_event)
            
            # Print real-time log
            print(f"🚦 CROSSING: Object #{object_id} crossed {direction_symbol} at {timestamp:.2f}s (x: {prev_x} → {x})")
        
        # Update position
        self.object_positions[object_id] = {
            'x': x,
            'y': y,
            'side': current_side,
            'last_timestamp': timestamp
        }
    
    def draw_middle_line(self, frame):
        """Draw the middle vertical line on the frame."""
        cv2.line(frame, (self.middle_line_x, 0), (self.middle_line_x, self.frame_height), 
                (0, 0, 255), 3)  # Red line
        
        # Add line label
        cv2.putText(frame, "CROSSING LINE", (self.middle_line_x + 10, 50), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    def get_statistics(self):
        """Get crossing detection statistics."""
        unique_objects = len(set(event['object_id'] for event in self.crossing_events))
        total_crossings = len(self.crossing_events)
        avg_crossings_per_object = total_crossings / unique_objects if unique_objects > 0 else 0
        
        return {
            'total_crossings': total_crossings,
            'unique_objects': unique_objects,
            'avg_crossings_per_object': avg_crossings_per_object,
            'crossing_events': self.crossing_events
        }


def get_track_color(track_id):
    """Get a consistent color for a track ID."""
    if track_id is None:
        return (128, 128, 128)  # Gray for untracked
    
    color_index = (track_id - 1) % len(BASE_COLORS)
    return BASE_COLORS[color_index]


def is_detection_in_tracking_zone(x1, y1, x2, y2, frame_width):
    """
    Check if a detection bounding box is in the tracking zone.
    
    Args:
        x1, y1, x2, y2: Bounding box coordinates
        frame_width: Width of the frame
        
    Returns:
        True if the detection center is in the tracking zone
    """
    center_x = (x1 + x2) / 2
    zone_x_min = frame_width * TRACK_ZONE_X_MIN
    zone_x_max = frame_width * TRACK_ZONE_X_MAX
    
    return zone_x_min <= center_x <= zone_x_max


def resize_frame(frame, target_width, target_height):
    """Resize frame to target dimensions."""
    return cv2.resize(frame, (target_width, target_height))


def draw_detections(frame, results, confidence_threshold=0.5, scale_x=1.0, scale_y=1.0):
    """
    Draw detections without tracking (fallback function).
    """
    annotated_frame = frame.copy()
    detection_count = 0
    
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        
        for box in boxes:
            # Get box coordinates and confidence with scaling
            coords = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = (coords * [scale_x, scale_y, scale_x, scale_y]).astype(int)
            conf = box.conf[0].cpu().numpy()
            cls = int(box.cls[0].cpu().numpy())
            
            # Only draw if confidence is above threshold
            if conf >= confidence_threshold:
                detection_count += 1
                
                # Draw bounding box
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Draw label
                label = f"{CLASS_NAMES[cls]}: {conf:.2f}"
                (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                cv2.rectangle(annotated_frame, (x1, y1 - label_h - 10), (x1 + label_w, y1), (0, 255, 0), -1)
                cv2.putText(annotated_frame, label, (x1, y1 - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    
    return annotated_frame, detection_count

print("✅ Utility classes loaded successfully!")
print("   • LineCrossingDetector: Detects objects crossing the middle line")
print("   • Helper functions: get_track_color, is_detection_in_tracking_zone, etc.")

## Step 5: Process Video with Live Detection

In [ ]:
def draw_detections_with_tracking(frame, results, confidence_threshold=0.5, scale_x=1.0, scale_y=1.0, line_detector=None):
    """
    Draw detections with tracking information.
    
    Args:
        frame: OpenCV frame (BGR)
        results: YOLO detection/tracking results
        confidence_threshold: Minimum confidence for displaying detections
        scale_x: X scaling factor for coordinates
        scale_y: Y scaling factor for coordinates
        line_detector: LineCrossingDetector instance for line crossing detection
    
    Returns:
        annotated_frame: Frame with drawn detections and tracking info
        detection_count: Number of detections above confidence threshold
        track_ids: List of active track IDs in this frame (YOLO IDs for tracking zone objects)
    """
    annotated_frame = frame.copy()
    detection_count = 0
    track_ids = []
    frame_height, frame_width = frame.shape[:2]
    
    # Draw middle line for line crossing detection
    if line_detector:
        line_detector.draw_middle_line(annotated_frame)
    
    # Draw tracking zone boundaries for visualization
    zone_x_min = int(frame_width * TRACK_ZONE_X_MIN)
    zone_x_max = int(frame_width * TRACK_ZONE_X_MAX)
    
    # Draw tracking zone as semi-transparent overlay
    overlay = annotated_frame.copy()
    cv2.rectangle(overlay, (zone_x_min, 0), (zone_x_max, frame_height), (0, 255, 255), -1)  # Yellow zone
    cv2.addWeighted(annotated_frame, 0.95, overlay, 0.05, 0, annotated_frame)
    
    # Draw zone boundary lines
    cv2.line(annotated_frame, (zone_x_min, 0), (zone_x_min, frame_height), (0, 255, 255), 2)
    cv2.line(annotated_frame, (zone_x_max, 0), (zone_x_max, frame_height), (0, 255, 255), 2)
    
    # Add zone labels
    cv2.putText(annotated_frame, "TRACKING ZONE", (zone_x_min + 10, 25), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        
        for box in boxes:
            # Get box coordinates, confidence, and class with scaling
            coords = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = (coords * [scale_x, scale_y, scale_x, scale_y]).astype(int)
            conf = box.conf[0].cpu().numpy()
            cls = int(box.cls[0].cpu().numpy())
            
            # Only draw if confidence is above threshold
            if conf >= confidence_threshold:
                detection_count += 1
                
                # Check if detection is in tracking zone
                in_tracking_zone = is_detection_in_tracking_zone(x1, y1, x2, y2, frame_width)
                
                # Get YOLO track ID if available
                yolo_track_id = None
                if hasattr(box, 'id') and box.id is not None:
                    yolo_track_id = int(box.id[0].cpu().numpy())
                    # Only add to track_ids if in tracking zone
                    if in_tracking_zone:
                        track_ids.append(yolo_track_id)
                
                # Get color based on YOLO track ID
                if yolo_track_id is not None and in_tracking_zone:
                    color = get_track_color(yolo_track_id)
                    line_thickness = 3  # Thicker lines for tracked objects
                else:
                    color = (128, 128, 128)  # Gray for untracked or out-of-zone detections
                    line_thickness = 2
                
                # Draw bounding box
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, line_thickness)
                
                # Create label with class name, confidence, and track ID
                if yolo_track_id is not None and in_tracking_zone:
                    label = f"ID:{yolo_track_id} {CLASS_NAMES[cls]}: {conf:.2f}"
                elif yolo_track_id is not None:
                    label = f"ID:{yolo_track_id} {CLASS_NAMES[cls]}: {conf:.2f} (OUT-OF-ZONE)"
                else:
                    label = f"{CLASS_NAMES[cls]}: {conf:.2f} (NO-TRACK)"
                
                # Get label size for background rectangle
                (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                
                # Draw label background
                cv2.rectangle(annotated_frame, (x1, y1 - label_h - 10), (x1 + label_w, y1), color, -1)
                
                # Draw label text
                cv2.putText(annotated_frame, label, (x1, y1 - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
                
                # Draw track ID prominently in the center of the box (only for tracked objects in zone)
                if yolo_track_id is not None and in_tracking_zone:
                    center_x = (x1 + x2) // 2
                    center_y = (y1 + y2) // 2
                    id_text = f"#{yolo_track_id}"
                    
                    # Get text size for centering
                    (id_w, id_h), _ = cv2.getTextSize(id_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 4)
                    
                    # Draw ID background circle
                    cv2.circle(annotated_frame, (center_x, center_y), max(id_w, id_h) // 2 + 8, color, -1)
                    
                    # Draw ID text
                    cv2.putText(annotated_frame, id_text, 
                               (center_x - id_w // 2, center_y + id_h // 2), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
    
    return annotated_frame, detection_count, track_ids


# Process video with tracking
print("🎬 Starting video processing with YOLO tracking...")
print(f"Using downscaling: Original {width}x{height} → Processing {process_width}x{process_height}")
print(f"Tracking enabled: {ENABLE_TRACKING}")
if ENABLE_TRACKING:
    print(f"Tracking zone: {TRACK_ZONE_X_MIN*100:.0f}% to {TRACK_ZONE_X_MAX*100:.0f}% of frame width")
    print(f"This helps avoid tracking issues when caps enter/exit frame edges")
start_time = time.time()

# Calculate scaling factors for coordinate conversion
scale_x = width / process_width
scale_y = height / process_height

# Open video for reading
cap = cv2.VideoCapture(VIDEO_PATH)

# Setup video writer if saving output
if SAVE_VIDEO:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

# Initialize frame counter and statistics
frame_number = 0
frames_with_detections = 0
total_detections = 0

# Tracking statistics
process_every_n_frames = 1  # Process every frame, increase to skip frames for speed
unique_tracks = set()  # Keep track of all unique YOLO track IDs
current_active_tracks = 0
max_simultaneous_tracks = 0

# Initialize line crossing detector
line_detector = LineCrossingDetector(width, height, fps) if ENABLE_TRACKING else None

if ENABLE_TRACKING and line_detector:
    print(f"Line crossing detection enabled - middle line at x={line_detector.middle_line_x}")

print(f"Processing {frame_count} frames...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_number += 1
    
    # Process every nth frame for performance
    if frame_number % process_every_n_frames == 0:
        # Resize frame for processing
        small_frame = resize_frame(frame, process_width, process_height)
        
        # Run YOLO detection with or without tracking
        if ENABLE_TRACKING:
            # Use tracking mode
            results = model.track(small_frame, verbose=False, tracker=TRACKER_TYPE, persist=True, device="mps")
        else:
            # Use detection only
            results = model(small_frame, verbose=False)
        
        # Draw detections on original frame with tracking info and scaling
        if ENABLE_TRACKING:
            annotated_frame, detection_count, frame_track_ids = draw_detections_with_tracking(
                frame, results, CONFIDENCE_THRESHOLD, scale_x, scale_y, line_detector)
            
            # Update tracking statistics
            unique_tracks.update(frame_track_ids)
            current_active_tracks = len(frame_track_ids)
            max_simultaneous_tracks = max(max_simultaneous_tracks, current_active_tracks)
            
            # Update line crossing detector with tracked objects
            if line_detector:
                # Calculate timestamp for this frame
                current_time = frame_number / fps
                
                # Update positions for all tracked objects in the tracking zone
                for track_id in frame_track_ids:
                    # Find the bounding box for this track ID
                    if len(results) > 0 and results[0].boxes is not None:
                        boxes = results[0].boxes
                        for box in boxes:
                            if hasattr(box, 'id') and box.id is not None:
                                yolo_track_id = int(box.id[0].cpu().numpy())
                                if yolo_track_id == track_id:
                                    # Get box coordinates with scaling
                                    coords = box.xyxy[0].cpu().numpy()
                                    x1, y1, x2, y2 = (coords * [scale_x, scale_y, scale_x, scale_y]).astype(int)
                                    center_x = (x1 + x2) // 2
                                    center_y = (y1 + y2) // 2
                                    
                                    # Check if this object is in the tracking zone
                                    frame_height, frame_width = frame.shape[:2]
                                    if is_detection_in_tracking_zone(x1, y1, x2, y2, frame_width):
                                        line_detector.update_position(track_id, center_x, center_y, current_time)
                                    break
        else:
            # Use original draw function for non-tracking mode
            annotated_frame, detection_count = draw_detections(frame, results, CONFIDENCE_THRESHOLD, scale_x, scale_y)
            frame_track_ids = []
        
        # Update statistics
        if detection_count > 0:
            frames_with_detections += 1
            total_detections += detection_count
        
        # Add frame info text
        if ENABLE_TRACKING:
            info_text = f"Frame: {frame_number}/{frame_count} | Detections: {detection_count} | YOLO IDs: {current_active_tracks}"
        else:
            info_text = f"Frame: {frame_number}/{frame_count} | Detections: {detection_count}"
        cv2.putText(annotated_frame, info_text, (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Add processing and tracking info
        if ENABLE_TRACKING:
            zone_info = f"Zone: {TRACK_ZONE_X_MIN*100:.0f}%-{TRACK_ZONE_X_MAX*100:.0f}%"
            tracking_info = f"YOLO IDs: {len(unique_tracks)} total"
            process_info = f"Processing: {process_width}x{process_height} | {tracking_info} | {zone_info}"
            if line_detector:
                crossings_info = f"Line crossings: {len(line_detector.crossing_events)} total"
                process_info += f" | {crossings_info}"
        else:
            process_info = f"Processing: {process_width}x{process_height} (scale: {1/scale_x:.2f}x)"
        cv2.putText(annotated_frame, process_info, (10, 60), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Save frame if requested
        if SAVE_FRAMES and detection_count > 0:  # Only save frames with detections
            frame_filename = f"{OUTPUT_FRAMES_DIR}/frame_{frame_number:04d}_det_{detection_count}.jpg"
            cv2.imwrite(frame_filename, annotated_frame)
        
        # Save to output video
        if SAVE_VIDEO:
            out.write(annotated_frame)
        
        # Display live (optional, can be slow)
        if DISPLAY_LIVE:
            # Show at reduced resolution for better performance
            display_frame = resize_frame(annotated_frame, process_width, process_height)
            cv2.imshow('Bottle Cap Detection with YOLO Tracking', display_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    else:
        # For skipped frames, just write original frame to output video
        if SAVE_VIDEO:
            out.write(frame)

# Clean up
cap.release()
if SAVE_VIDEO:
    out.release()
cv2.destroyAllWindows()

# Processing summary
end_time = time.time()
processing_time = end_time - start_time
fps_processed = frame_number / processing_time if processing_time > 0 else 0

print("\n" + "="*80)
print("🎯 PROCESSING COMPLETE - VALIDATION SUMMARY")
print("="*80)

print(f"\n📊 DETECTION STATISTICS:")
print(f"   • Total frames processed: {frame_number:,}")
print(f"   • Frames with detections: {frames_with_detections:,}")
print(f"   • Total detections: {total_detections:,}")
print(f"   • Average detections per frame: {total_detections/frame_number:.2f}")
print(f"   • Detection rate: {frames_with_detections/frame_number*100:.1f}%")

if ENABLE_TRACKING:
    print(f"\n🎯 TRACKING STATISTICS:")
    print(f"   • Unique YOLO track IDs detected: {len(unique_tracks)}\")\n    print(f\"   • Max simultaneous YOLO tracks: {max_simultaneous_tracks}")
    
    # Get ID mapper statistics
    if id_mapper:
        mapper_stats = id_mapper.get_statistics()
        print(f"   • ID mappings created: {mapper_stats['total_mappings']}")
        print(f"   • Currently active mappings: {mapper_stats['active_mappings']}")
        print(f"   • Next custom ID to assign: {mapper_stats['next_custom_id']}")
    
    # Line crossing statistics
    if line_detector:
        print(f"\n🚦 LINE CROSSING STATISTICS:")
        crossing_stats = line_detector.get_statistics()
        print(f"   • Total line crossings detected: {crossing_stats['total_crossings']}")
        print(f"   • Objects that crossed the line: {crossing_stats['unique_objects']}")
        print(f"   • Average crossings per object: {crossing_stats['avg_crossings_per_object']:.1f}")
        
        if crossing_stats['total_crossings'] > 0:
            print(f"\n📝 CROSSING EVENTS LOG:")
            for i, event in enumerate(line_detector.crossing_events[-10:], 1):  # Show last 10 events
                direction = "→" if event['direction'] == 'left_to_right' else "←"
                print(f"   {i:2d}. Object #{event['object_id']} crossed {direction} at {event['timestamp']:.2f}s")
            
            if len(line_detector.crossing_events) > 10:
                print(f"   ... and {len(line_detector.crossing_events) - 10} earlier events")

print(f"\n⚡ PERFORMANCE:")
print(f"   • Processing time: {processing_time:.1f} seconds")
print(f"   • Processing FPS: {fps_processed:.1f}")
print(f"   • Original video FPS: {fps:.1f}")
print(f"   • Speed ratio: {fps_processed/fps:.2f}x real-time")

if SAVE_VIDEO:
    file_size = os.path.getsize(OUTPUT_VIDEO_PATH) / (1024*1024)  # MB
    print(f"\n💾 OUTPUT:")
    print(f"   • Video saved: {OUTPUT_VIDEO_PATH}")
    print(f"   • File size: {file_size:.1f} MB")

print("\n✅ Validation complete! The custom sequential tracking IDs provide clean numbering")
print("   while maintaining YOLO's robust tracking accuracy. ID gaps in YOLO tracking")
print("   are normal and indicate healthy selective tracking behavior.")

## Step 6: Display Results and Statistics

In [ ]:
# Display detection statistics
print("=== DETECTION STATISTICS ===")
print(f"Video: {VIDEO_PATH}")
print(f"Model: {MODEL_PATH}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Tracking enabled: {ENABLE_TRACKING}")
if ENABLE_TRACKING:
    print(f"Tracking zone: {TRACK_ZONE_X_MIN*100:.0f}% to {TRACK_ZONE_X_MAX*100:.0f}% of frame width")
    print(f"YOLO tracking: ENABLED (using native YOLO track IDs only)")

print(f"\nResults:")
print(f"  • Total frames: {frame_count}")
print(f"  • Frames with detections: {frames_with_detections} ({frames_with_detections/frame_count*100:.1f}%)")
print(f"  • Total bottle caps detected: {total_detections}")
print(f"  • Average detections per frame: {total_detections/frame_count:.2f}")

if ENABLE_TRACKING and 'unique_tracks' in locals():
    print(f"\n🎯 TRACKING STATISTICS:")
    print(f"  • Tracking zone: {TRACK_ZONE_X_MIN*100:.0f}%-{TRACK_ZONE_X_MAX*100:.0f}% of frame width (avoids edge issues)")
    
    # YOLO tracking stats
    print(f"\n📊 YOLO Tracking (Internal System):")
    print(f"  • Unique YOLO tracks discovered: {len(unique_tracks)}")
    print(f"  • Maximum simultaneous YOLO tracks: {max_simultaneous_tracks}")
    if unique_tracks:
        print(f"  • YOLO track IDs: {sorted(list(unique_tracks))}")
        print(f"  • Note: YOLO IDs may have gaps due to brief/failed detections")
    
    # Custom ID stats
    if 'unique_custom_ids' in locals() and unique_custom_ids:
        print(f"\n🔢 Custom Sequential IDs (Visualization):")
        print(f"  • Custom IDs assigned: {len(unique_custom_ids)}")
        print(f"  • Maximum simultaneous custom IDs: {max_simultaneous_custom_ids}")
        print(f"  • Custom IDs used: {sorted(list(unique_custom_ids))}")
        print(f"  • ✅ Sequential numbering: 1, 2, 3, 4... (no gaps!)")
        print(f"  • Custom IDs provide clean visualization while preserving YOLO tracking accuracy")
    
    print(f"  • Note: Objects outside tracking zone are detected but not given stable IDs")

# Display file sizes
if os.path.exists(OUTPUT_VIDEO_PATH):
    output_size = os.path.getsize(OUTPUT_VIDEO_PATH) / (1024 * 1024)  # MB
    print(f"\n📁 Output video size: {output_size:.2f} MB")

if SAVE_FRAMES and os.path.exists(OUTPUT_FRAMES_DIR):
    frame_files = [f for f in os.listdir(OUTPUT_FRAMES_DIR) if f.endswith('.jpg')]
    print(f"📁 Saved {len(frame_files)} annotated frames")